In [2]:
# Primero cargamos nuestro archivo
from google.colab import files
uploaded = files.upload()

Saving patients_diabetis - copia.txt to patients_diabetis - copia.txt


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Cargar el archivo (ajusta el nombre si es necesario)
df = pd.read_csv('patients_diabetis - copia.txt', sep='\t')

# 1. Separar predictores y variable objetivo
# En diabetes, la columna suele llamarse 'class' o 'Outcome'
X = df.drop(columns=['class'])
y = df['class']

# 2. División de datos según tus parámetros
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.33,
    train_size=0.67,
    random_state=42
)

# Verificar la división
print(f"Registros entrenamiento: {len(X_train)}")
print(f"Registros test: {len(X_test)}")

Registros entrenamiento: 514
Registros test: 254


In [6]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import SGD
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1. Preprocesamiento
# Escalar los datos es obligatorio para ANN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convertir etiquetas a números
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

# --- ARQUITECTURA 1: (12, 6) ---
model_12_6 = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(12, activation='relu', kernel_initializer='he_uniform'),
    Dense(6, activation='relu', kernel_initializer='he_uniform'),
    Dense(1, activation='sigmoid', kernel_initializer='glorot_uniform')
])

model_12_6.compile(optimizer=SGD(learning_rate=0.01), loss='binary_crossentropy', metrics=['accuracy'])
print("Entrenando arquitectura 12-6...")
history1 = model_12_6.fit(X_train_scaled, y_train_encoded, validation_data=(X_test_scaled, y_test_encoded), epochs=100, verbose=0)

# --- ARQUITECTURA 2: (20, 10) ---
model_20_10 = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(20, activation='relu', kernel_initializer='he_uniform'),
    Dense(10, activation='relu', kernel_initializer='he_uniform'),
    Dense(1, activation='sigmoid', kernel_initializer='glorot_uniform')
])

model_20_10.compile(optimizer=SGD(learning_rate=0.01), loss='binary_crossentropy', metrics=['accuracy'])
print("Entrenando arquitectura 20-10...")
history2 = model_20_10.fit(X_train_scaled, y_train_encoded, validation_data=(X_test_scaled, y_test_encoded), epochs=100, verbose=0)

# --- ARQUITECTURA 3: (20, 10, 5) ---
model_20_10_5 = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(20, activation='relu', kernel_initializer='he_uniform'),
    Dense(10, activation='relu', kernel_initializer='he_uniform'),
    Dense(5, activation='relu', kernel_initializer='he_uniform'),
    Dense(1, activation='sigmoid', kernel_initializer='glorot_uniform')
])

model_20_10_5.compile(optimizer=SGD(learning_rate=0.01), loss='binary_crossentropy', metrics=['accuracy'])
print("Entrenando arquitectura 20-10-5...")
history3 = model_20_10_5.fit(X_train_scaled, y_train_encoded, validation_data=(X_test_scaled, y_test_encoded), epochs=100, verbose=0)

Entrenando arquitectura 12-6...
Entrenando arquitectura 20-10...
Entrenando arquitectura 20-10-5...


In [9]:
from sklearn.metrics import confusion_matrix, accuracy_score, cohen_kappa_score
import pandas as pd
import numpy as np

def obtener_metricas(modelo, X_test, y_test, nombre_config):
    # 1. Obtener predicciones (probabilidades) y convertirlas a clases (0 o 1)
    probabilidades = modelo.predict(X_test, verbose=0)
    predicciones = (probabilidades > 0.5).astype(int)

    # 2. Calcular Matriz de Confusión
    cm = confusion_matrix(y_test, predicciones)

    # Extraer valores de la matriz (TN, FP, FN, TP)
    tn, fp, fn, tp = cm.ravel()

    # 3. Calcular métricas
    acc = accuracy_score(y_test, predicciones)
    kappa = cohen_kappa_score(y_test, predicciones)
    sens = tp / (tp + fn)  # Sensibilidad (Recall)
    spec = tn / (tn + fp)  # Especificidad

    return {
        "Algoritmo": "ANN",
        "Configuracion": nombre_config,
        "Accuracy": round(acc, 4),
        "Kappa": round(kappa, 4),
        "Sensitivity": round(sens, 4),
        "Specificity": round(spec, 4)
    }

# 4. Generar la lista de resultados
resultados_finales = []
resultados_finales.append(obtener_metricas(model_12_6, X_test_scaled, y_test_encoded, "12-6"))
resultados_finales.append(obtener_metricas(model_20_10, X_test_scaled, y_test_encoded, "20-10"))
resultados_finales.append(obtener_metricas(model_20_10_5, X_test_scaled, y_test_encoded, "20-10-5"))

# 5. Crear el DataFrame comparativo
comparativa_ann = pd.DataFrame(resultados_finales)
print(comparativa_ann)

  Algoritmo Configuracion  Accuracy   Kappa  Sensitivity  Specificity
0       ANN          12-6    0.7638  0.4572       0.5930       0.8512
1       ANN         20-10    0.7480  0.4437       0.6512       0.7976
2       ANN       20-10-5    0.7402  0.4098       0.5814       0.8214
